In [ ]:
import pickle
import torch
import torch.nn.functional as F

# Loading the embeddings
mixed_emb_path = "embeddings/mixed_emb.pkl"
sentence_emb_path = "embeddings/sentence_transformers_emb.pkl"
node2vec_emb_path = "embeddings/node2vec_emb.pkl"

node2vec_emb_dict = {}
with open(node2vec_emb_path, "rb") as fIn:
    node2vec_emb_dict = torch.load(fIn)

mixed_emb_dict = {}
with open(mixed_emb_path, "rb") as fIn:
    mixed_emb_dict = torch.load(fIn)

sentence_emb_dict = {}
with open(sentence_emb_path, "rb") as fIn:
    sentence_emb_dict = pickle.load(fIn)

C:\Users\alelc\AppData\Local\Temp\ipykernel_24600\877676200.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  node2vec_emb_dict = torch.load(fIn)
C:\Users\alelc\AppData\L

In [3]:
print(list(node2vec_emb_dict.keys())[:5])

['pawoo.net', 'baraag.net', 'aethy.com', 'mastodon.social', 'journa.host']


In [ ]:
import pandas as pd

# Loading the dataset
dataset = pd.read_csv('data/Mastodon_instances_edges_Sept2024_filtered.csv')
dataset.head()

,source_domain,target_domain,weight
0,pawoo.net,baraag.net,305426
1,baraag.net,pawoo.net,298256
2,aethy.com,baraag.net,195144
3,baraag.net,misskey.io,194807
4,mastodon.social,journa.host,176396


In [ ]:
import torch
import tqdm

json_data = []

# Creating the JSON data
for source, target, _ in tqdm.tqdm(dataset.itertuples(index=False), total=len(dataset)):

    if source in node2vec_emb_dict and target in node2vec_emb_dict:
        source_emb = torch.Tensor(node2vec_emb_dict[source])
        target_emb = torch.Tensor(node2vec_emb_dict[target])
        node2vec_sim = F.cosine_similarity(source_emb, target_emb, dim=0).tolist()
        
        source_st_emb = torch.Tensor(sentence_emb_dict[source])
        target_st_emb = torch.Tensor(sentence_emb_dict[target])
        sentence_sim = F.cosine_similarity(source_st_emb, target_st_emb, dim=0).tolist()

        source_mx_emb = torch.Tensor(mixed_emb_dict[source])
        target_mx_emb = torch.Tensor(mixed_emb_dict[target])
        mixed_sim = F.cosine_similarity(source_mx_emb, target_mx_emb, dim=0).tolist()

        entry = {
            "source": source,
            "target": target,
            "node2vec_similarity": node2vec_sim,
            "description_similarity": sentence_sim,
            "mixed_similarity": mixed_sim
        }
        json_data.append(entry)

100%|██████████| 217879/217879 [00:12<00:00, 17310.67it/s]


In [8]:
print(json_data[:5])

[{'source': 'pawoo.net', 'target': 'baraag.net', 'node2vec_similarity': 0.39055678248405457, 'description_similarity': 0.14394132792949677, 'mixed_similarity': 0.3757435083389282}, {'source': 'baraag.net', 'target': 'pawoo.net', 'node2vec_similarity': 0.39055678248405457, 'description_similarity': 0.14394132792949677, 'mixed_similarity': 0.3757435083389282}, {'source': 'aethy.com', 'target': 'baraag.net', 'node2vec_similarity': 0.3295334577560425, 'description_similarity': 0.550489068031311, 'mixed_similarity': 0.3671788275241852}, {'source': 'mastodon.social', 'target': 'journa.host', 'node2vec_similarity': 0.051246561110019684, 'description_similarity': 0.5087612271308899, 'mixed_similarity': 0.05056322365999222}, {'source': 'mastodon.social', 'target': 'hachyderm.io', 'node2vec_similarity': 0.9470502138137817, 'description_similarity': 0.3666834831237793, 'mixed_similarity': 0.9464789628982544}]


In [9]:
import json

json_output = json.dumps(json_data, indent=4)

with open("similarities.json", "w") as f:
    f.write(json_output)